# Data Lake Storage Design and Delta Lake Bridge

## File Formats, Partitioning, Small Files, and Reliable Lakehouse Tables

This notebook continues after the Data Lake introduction.

You will learn how a Data Lake becomes faster, better organized, and more reliable.

## Topics covered

- CSV vs JSON vs Parquet
- Row-based vs columnar storage
- Data Lake partitioning
- Partition pruning
- Good and bad partition columns
- Small files problem
- Why partitioning alone is not enough
- Why Delta Lake exists
- Delta Lake concepts: transaction log, ACID, schema enforcement, time travel, update, delete, merge
- Bronze, Silver, Gold with Delta Lake


# 1. Data Lake Recap

A Data Lake stores raw and processed data files at scale.

A common structure is:

```text
data_lake/
  bronze/
  silver/
  gold/
```

## Bronze Layer

Raw data as received from sources.

## Silver Layer

Cleaned and standardized data.

## Gold Layer

Business-ready analytics data.

<svg width="900" height="220" xmlns="http://www.w3.org/2000/svg">
  <rect x="40" y="70" width="160" height="70" rx="10" fill="#eef4ff" stroke="#3b5bdb"/>
  <text x="120" y="100" text-anchor="middle" font-size="16" font-family="Arial">Source Data</text>
  <text x="120" y="122" text-anchor="middle" font-size="12" font-family="Arial">CSV, JSON, Logs</text>
  <rect x="260" y="70" width="150" height="70" rx="10" fill="#fff4e6" stroke="#f08c00"/>
  <text x="335" y="100" text-anchor="middle" font-size="16" font-family="Arial">Bronze</text>
  <text x="335" y="122" text-anchor="middle" font-size="12" font-family="Arial">Raw files</text>
  <rect x="470" y="70" width="150" height="70" rx="10" fill="#f1f3f5" stroke="#495057"/>
  <text x="545" y="100" text-anchor="middle" font-size="16" font-family="Arial">Silver</text>
  <text x="545" y="122" text-anchor="middle" font-size="12" font-family="Arial">Clean data</text>
  <rect x="680" y="70" width="150" height="70" rx="10" fill="#ebfbee" stroke="#2b8a3e"/>
  <text x="755" y="100" text-anchor="middle" font-size="16" font-family="Arial">Gold</text>
  <text x="755" y="122" text-anchor="middle" font-size="12" font-family="Arial">Analytics</text>
  <line x1="200" y1="105" x2="260" y2="105" stroke="#333" stroke-width="2" marker-end="url(#a)"/>
  <line x1="410" y1="105" x2="470" y2="105" stroke="#333" stroke-width="2" marker-end="url(#a)"/>
  <line x1="620" y1="105" x2="680" y2="105" stroke="#333" stroke-width="2" marker-end="url(#a)"/>
  <defs><marker id="a" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#333"/></marker></defs>
</svg>

## Main idea

```text
Data Lake structure gives organization.
File formats and partitioning improve performance.
Delta Lake adds reliability.
```


# 2. Why File Format Matters

A Data Lake is made of files.

The file format affects:

```text
storage size
read speed
write speed
schema handling
compression
analytics performance
```

Common formats:

| Format | Common Layer | Best For |
|---|---|---|
| CSV | Bronze | Simple tabular source data |
| JSON | Bronze | API and event data |
| Parquet | Silver and Gold | Analytics and large-scale processing |
| Delta | Silver and Gold | Reliable lakehouse tables |


# 3. Row-Based vs Columnar Storage

CSV and JSON are usually row-based.

Parquet is columnar.

## Row-based idea

```text
Row 1: order_id, city, amount, status
Row 2: order_id, city, amount, status
```

## Columnar idea

```text
order_id column stored together
city column stored together
amount column stored together
status column stored together
```

<svg width="900" height="260" xmlns="http://www.w3.org/2000/svg">
  <text x="200" y="35" text-anchor="middle" font-size="18" font-family="Arial">Row-Based Storage</text>
  <rect x="40" y="60" width="320" height="45" fill="#eef4ff" stroke="#3b5bdb"/>
  <text x="200" y="88" text-anchor="middle" font-size="14" font-family="Arial">1001, Delhi, 2500, completed</text>
  <rect x="40" y="105" width="320" height="45" fill="#eef4ff" stroke="#3b5bdb"/>
  <text x="200" y="133" text-anchor="middle" font-size="14" font-family="Arial">1002, Mumbai, 1200, pending</text>
  <rect x="40" y="150" width="320" height="45" fill="#eef4ff" stroke="#3b5bdb"/>
  <text x="200" y="178" text-anchor="middle" font-size="14" font-family="Arial">1003, Pune, 3200, completed</text>
  <text x="670" y="35" text-anchor="middle" font-size="18" font-family="Arial">Columnar Storage</text>
  <rect x="470" y="60" width="90" height="150" fill="#fff4e6" stroke="#f08c00"/>
  <text x="515" y="88" text-anchor="middle" font-size="13" font-family="Arial">order_id</text>
  <text x="515" y="120" text-anchor="middle" font-size="13" font-family="Arial">1001</text>
  <text x="515" y="145" text-anchor="middle" font-size="13" font-family="Arial">1002</text>
  <text x="515" y="170" text-anchor="middle" font-size="13" font-family="Arial">1003</text>
  <rect x="570" y="60" width="90" height="150" fill="#ebfbee" stroke="#2b8a3e"/>
  <text x="615" y="88" text-anchor="middle" font-size="13" font-family="Arial">city</text>
  <text x="615" y="120" text-anchor="middle" font-size="13" font-family="Arial">Delhi</text>
  <text x="615" y="145" text-anchor="middle" font-size="13" font-family="Arial">Mumbai</text>
  <text x="615" y="170" text-anchor="middle" font-size="13" font-family="Arial">Pune</text>
  <rect x="670" y="60" width="90" height="150" fill="#f1f3f5" stroke="#495057"/>
  <text x="715" y="88" text-anchor="middle" font-size="13" font-family="Arial">amount</text>
  <text x="715" y="120" text-anchor="middle" font-size="13" font-family="Arial">2500</text>
  <text x="715" y="145" text-anchor="middle" font-size="13" font-family="Arial">1200</text>
  <text x="715" y="170" text-anchor="middle" font-size="13" font-family="Arial">3200</text>
  <rect x="770" y="60" width="100" height="150" fill="#fff0f6" stroke="#c2255c"/>
  <text x="820" y="88" text-anchor="middle" font-size="13" font-family="Arial">status</text>
  <text x="820" y="120" text-anchor="middle" font-size="13" font-family="Arial">completed</text>
  <text x="820" y="145" text-anchor="middle" font-size="13" font-family="Arial">pending</text>
  <text x="820" y="170" text-anchor="middle" font-size="13" font-family="Arial">completed</text>
</svg>

If a query only needs `city` and `amount`, Parquet can avoid reading unnecessary columns.


# 4. Setup PySpark

Run the next cell to install PySpark.


In [ ]:
!pip install pyspark -q


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lower, trim, to_date, sum as spark_sum,
    count, avg, desc, current_timestamp, lit, when
)
import os
import shutil
from pathlib import Path
import json
import csv

spark = (
    SparkSession.builder
    .appName("DataLakeStorageAndDeltaBridge")
    .getOrCreate()
)

spark


# 5. Create E-Commerce Source Data

We will create source data for:

```text
orders.csv
customers.csv
products.json
payments.csv
clickstream_events.json
```

This data simulates a small e-commerce company.


In [ ]:
base_path = Path("datalake_storage_delta_bridge")

if base_path.exists():
    shutil.rmtree(base_path)

source_path = base_path / "source"
lake_path = base_path / "lake"

source_path.mkdir(parents=True, exist_ok=True)
lake_path.mkdir(parents=True, exist_ok=True)

bronze_path = lake_path / "bronze"
silver_path = lake_path / "silver"
gold_path = lake_path / "gold"

for path in [bronze_path, silver_path, gold_path]:
    path.mkdir(parents=True, exist_ok=True)

print("Base path:", base_path)


Base path: datalake_storage_delta_bridge


In [ ]:
orders_rows = [
    ["order_id", "customer_id", "product_id", "order_date", "amount", "status", "city", "country"],
    [1001, "C001", "P001", "2026-08-10", 2500, "Completed", "Delhi", "India"],
    [1002, "C002", "P002", "2026-08-10", -500, "completed", "Mumbai", "India"],
    [1003, "C003", "P003", "2026-08-10", "", "Pending", "Delhi", "India"],
    [1004, "C004", "P004", "2026-08-11", 3200, "Completed", "Pune", "India"],
    [1005, "C005", "P001", "2026-08-11", 1200, "Cancelled", "Mumbai", "India"],
    [1006, "", "P002", "2026-08-11", 1800, "Completed", "Chennai", "India"],
    [1007, "C002", "P005", "2026-08-12", 4500, "completed", "Bangalore", "India"],
    [1008, "C003", "P003", "2026-08-12", 900, "FAILED", "Delhi", "India"],
    [1009, "C006", "P006", "2026-08-12", 7500, "Completed", "Dubai", "UAE"],
    [1010, "C007", "P007", "2026-08-13", 6200, "Completed", "Singapore", "Singapore"],
    [1011, "C008", "P001", "2026-08-13", 2400, "completed", "Delhi", "India"],
    [1001, "C001", "P001", "2026-08-10", 2500, "Completed", "Delhi", "India"],
]

orders_file = source_path / "orders.csv"
with open(orders_file, "w", newline="") as f:
    csv.writer(f).writerows(orders_rows)

customers_rows = [
    ["customer_id", "customer_name", "segment", "signup_date", "country"],
    ["C001", "Aarav Sharma", "Premium", "2025-01-15", "India"],
    ["C002", "Meera Iyer", "Standard", "2025-03-20", "India"],
    ["C003", "Kabir Khan", "Premium", "2025-05-01", "India"],
    ["C004", "Ananya Rao", "Standard", "2025-07-11", "India"],
    ["C005", "Rohan Gupta", "New", "2026-01-01", "India"],
    ["C006", "Sara Ali", "Premium", "2026-02-14", "UAE"],
    ["C007", "Liam Tan", "Standard", "2026-03-05", "Singapore"],
    ["C008", "Nisha Verma", "Premium", "2026-04-09", "India"],
]
customers_file = source_path / "customers.csv"
with open(customers_file, "w", newline="") as f:
    csv.writer(f).writerows(customers_rows)

products_data = [
    {"product_id": "P001", "product_name": "Laptop", "category": "Electronics", "price": 2500},
    {"product_id": "P002", "product_name": "Headphones", "category": "Electronics", "price": 1800},
    {"product_id": "P003", "product_name": "Backpack", "category": "Fashion", "price": 900},
    {"product_id": "P004", "product_name": "Office Chair", "category": "Furniture", "price": 3200},
    {"product_id": "P005", "product_name": "Smart Watch", "category": "Electronics", "price": 4500},
    {"product_id": "P006", "product_name": "Gaming Console", "category": "Electronics", "price": 7500},
    {"product_id": "P007", "product_name": "Tablet", "category": "Electronics", "price": 6200},
]
products_file = source_path / "products.json"
with open(products_file, "w") as f:
    for item in products_data:
        f.write(json.dumps(item) + "")

payments_rows = [
    ["payment_id", "order_id", "payment_method", "payment_status", "payment_date"],
    ["PMT001", 1001, "UPI", "success", "2026-08-10"],
    ["PMT002", 1002, "card", "failed", "2026-08-10"],
    ["PMT003", 1004, "netbanking", "success", "2026-08-11"],
    ["PMT004", 1007, "UPI", "success", "2026-08-12"],
    ["PMT005", 1008, "card", "failed", "2026-08-12"],
    ["PMT006", 1009, "card", "success", "2026-08-12"],
    ["PMT007", 1010, "wallet", "success", "2026-08-13"],
    ["PMT008", 1011, "UPI", "success", "2026-08-13"],
]
payments_file = source_path / "payments.csv"
with open(payments_file, "w", newline="") as f:
    csv.writer(f).writerows(payments_rows)

clickstream_data = [
    {"event_id": "E001", "customer_id": "C001", "event_time": "2026-08-10 10:01:00", "event_type": "view", "product_id": "P001"},
    {"event_id": "E002", "customer_id": "C001", "event_time": "2026-08-10 10:03:00", "event_type": "add_to_cart", "product_id": "P001"},
    {"event_id": "E003", "customer_id": "C001", "event_time": "2026-08-10 10:05:00", "event_type": "purchase", "product_id": "P001"},
    {"event_id": "E004", "customer_id": "C002", "event_time": "2026-08-10 11:00:00", "event_type": "view", "product_id": "P002"},
    {"event_id": "E005", "customer_id": "C003", "event_time": "2026-08-11 09:30:00", "event_type": "view", "product_id": "P003"},
    {"event_id": "E006", "customer_id": "C004", "event_time": "2026-08-11 12:10:00", "event_type": "purchase", "product_id": "P004"},
    {"event_id": "E007", "customer_id": "C002", "event_time": "2026-08-12 15:45:00", "event_type": "purchase", "product_id": "P005"},
    {"event_id": "E008", "customer_id": "C007", "event_time": "2026-08-13 16:15:00", "event_type": "purchase", "product_id": "P007"},
]
clickstream_file = source_path / "clickstream_events.json"
with open(clickstream_file, "w") as f:
    for event in clickstream_data:
        f.write(json.dumps(event) + "")

print("Source files created:")
for p in source_path.iterdir():
    print(p)


Source files created:
datalake_storage_delta_bridge/source/clickstream_events.json
datalake_storage_delta_bridge/source/customers.csv
datalake_storage_delta_bridge/source/products.json
datalake_storage_delta_bridge/source/orders.csv
datalake_storage_delta_bridge/source/payments.csv


# 6. Bronze Layer: Raw Files

Bronze keeps source files close to their original form.

This gives traceability and allows pipelines to be replayed later.


In [ ]:
ingestion_date = "2026-08-12"

bronze_sources = {
    "orders": orders_file,
    "customers": customers_file,
    "products": products_file,
    "payments": payments_file,
    "clickstream_events": clickstream_file,
}

for dataset_name, local_file in bronze_sources.items():
    target_dir = bronze_path / dataset_name / f"ingestion_date={ingestion_date}"
    target_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(local_file, target_dir / local_file.name)

for path in bronze_path.rglob("*"):
    if path.is_file():
        print(path)


datalake_storage_delta_bridge/lake/bronze/orders/ingestion_date=2026-08-12/orders.csv
datalake_storage_delta_bridge/lake/bronze/products/ingestion_date=2026-08-12/products.json
datalake_storage_delta_bridge/lake/bronze/payments/ingestion_date=2026-08-12/payments.csv
datalake_storage_delta_bridge/lake/bronze/clickstream_events/ingestion_date=2026-08-12/clickstream_events.json
datalake_storage_delta_bridge/lake/bronze/customers/ingestion_date=2026-08-12/customers.csv


# 7. Read and Clean Bronze Data

Bronze data may have duplicates, missing values, invalid records, and inconsistent text.

Silver data is cleaned and standardized.


In [ ]:
bronze_orders = spark.read.csv(str(bronze_path / "orders" / f"ingestion_date={ingestion_date}" / "orders.csv"), header=True, inferSchema=True)
bronze_customers = spark.read.csv(str(bronze_path / "customers" / f"ingestion_date={ingestion_date}" / "customers.csv"), header=True, inferSchema=True)
bronze_products = spark.read.json(str(bronze_path / "products" / f"ingestion_date={ingestion_date}" / "products.json"))
bronze_payments = spark.read.csv(str(bronze_path / "payments" / f"ingestion_date={ingestion_date}" / "payments.csv"), header=True, inferSchema=True)
bronze_clickstream = spark.read.json(str(bronze_path / "clickstream_events" / f"ingestion_date={ingestion_date}" / "clickstream_events.json"))

bronze_orders.show()


+--------+-----------+----------+----------+------+---------+---------+---------+
|order_id|customer_id|product_id|order_date|amount|   status|     city|  country|
+--------+-----------+----------+----------+------+---------+---------+---------+
|    1001|       C001|      P001|2026-08-10|  2500|Completed|    Delhi|    India|
|    1002|       C002|      P002|2026-08-10|  -500|completed|   Mumbai|    India|
|    1003|       C003|      P003|2026-08-10|  NULL|  Pending|    Delhi|    India|
|    1004|       C004|      P004|2026-08-11|  3200|Completed|     Pune|    India|
|    1005|       C005|      P001|2026-08-11|  1200|Cancelled|   Mumbai|    India|
|    1006|       NULL|      P002|2026-08-11|  1800|Completed|  Chennai|    India|
|    1007|       C002|      P005|2026-08-12|  4500|completed|Bangalore|    India|
|    1008|       C003|      P003|2026-08-12|   900|   FAILED|    Delhi|    India|
|    1009|       C006|      P006|2026-08-12|  7500|Completed|    Dubai|      UAE|
|    1010|      

In [ ]:
silver_orders = (
    bronze_orders
    .dropDuplicates(["order_id"])
    .filter(col("customer_id").isNotNull())
    .filter(col("amount").isNotNull())
    .filter(col("amount") > 0)
    .withColumn("status", lower(trim(col("status"))))
    .withColumn("city", trim(col("city")))
    .withColumn("country", trim(col("country")))
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("processed_at", current_timestamp())
)

silver_customers = (
    bronze_customers
    .dropDuplicates(["customer_id"])
    .withColumn("segment", lower(trim(col("segment"))))
    .withColumn("signup_date", to_date(col("signup_date")))
)

silver_products = (
    bronze_products
    .dropDuplicates(["product_id"])
    .withColumn("category", lower(trim(col("category"))))
)

silver_payments = (
    bronze_payments
    .dropDuplicates(["payment_id"])
    .withColumn("payment_status", lower(trim(col("payment_status"))))
    .withColumn("payment_method", lower(trim(col("payment_method"))))
    .withColumn("payment_date", to_date(col("payment_date")))
)

silver_clickstream = (
    bronze_clickstream
    .dropDuplicates(["event_id"])
    .withColumn("event_type", lower(trim(col("event_type"))))
    .withColumn("event_time", col("event_time").cast("timestamp"))
)

silver_orders.show()
silver_orders.printSchema()


+--------+-----------+----------+----------+------+---------+---------+---------+--------------------+
|order_id|customer_id|product_id|order_date|amount|   status|     city|  country|        processed_at|
+--------+-----------+----------+----------+------+---------+---------+---------+--------------------+
|    1001|       C001|      P001|2026-08-10|  2500|completed|    Delhi|    India|2026-08-12 12:25:...|
|    1004|       C004|      P004|2026-08-11|  3200|completed|     Pune|    India|2026-08-12 12:25:...|
|    1005|       C005|      P001|2026-08-11|  1200|cancelled|   Mumbai|    India|2026-08-12 12:25:...|
|    1007|       C002|      P005|2026-08-12|  4500|completed|Bangalore|    India|2026-08-12 12:25:...|
|    1008|       C003|      P003|2026-08-12|   900|   failed|    Delhi|    India|2026-08-12 12:25:...|
|    1009|       C006|      P006|2026-08-12|  7500|completed|    Dubai|      UAE|2026-08-12 12:25:...|
|    1010|       C007|      P007|2026-08-13|  6200|completed|Singapore|Si

# 8. CSV vs JSON vs Parquet Demo

The same dataset will be written in three formats:

```text
CSV
JSON
Parquet
```

Parquet is usually preferred for Silver and Gold because it is compressed and columnar.


In [ ]:
format_demo_path = base_path / "format_demo"
csv_output = format_demo_path / "orders_csv"
json_output = format_demo_path / "orders_json"
parquet_output = format_demo_path / "orders_parquet"

silver_orders.write.mode("overwrite").csv(str(csv_output), header=True)
silver_orders.write.mode("overwrite").json(str(json_output))
silver_orders.write.mode("overwrite").parquet(str(parquet_output))

print("Wrote CSV, JSON, and Parquet outputs.")


Wrote CSV, JSON, and Parquet outputs.


In [ ]:
def folder_size_bytes(path):
    total = 0
    path = Path(path)
    for file_path in path.rglob("*"):
        if file_path.is_file():
            total += file_path.stat().st_size
    return total

for fmt, path in [("CSV", csv_output), ("JSON", json_output), ("Parquet", parquet_output)]:
    print(f"{fmt}: {folder_size_bytes(path)} bytes")


CSV: 737 bytes
JSON: 1598 bytes
Parquet: 3045 bytes


In [ ]:
parquet_orders = spark.read.parquet(str(parquet_output))
selected_columns = parquet_orders.select("city", "amount")
selected_columns.show()
selected_columns.explain()


+---------+------+
|     city|amount|
+---------+------+
|    Delhi|  2500|
|     Pune|  3200|
|   Mumbai|  1200|
|Bangalore|  4500|
|    Delhi|   900|
|    Dubai|  7500|
|Singapore|  6200|
|    Delhi|  2400|
+---------+------+

== Physical Plan ==
*(1) Project [city#487, amount#485]
+- *(1) ColumnarToRow
   +- FileScan parquet [amount#485,city#487] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/datalake_storage_delta_bridge/format_demo/orders_parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<amount:int,city:string>




# 9. What is Partitioning?

Partitioning means physically organizing data into folders based on column values.

Example:

```text
silver/orders/
  order_date=2026-08-10/
  order_date=2026-08-11/
  order_date=2026-08-12/
```

If a query filters for one date, Spark can skip unrelated folders.

<svg width="900" height="300" xmlns="http://www.w3.org/2000/svg">
  <rect x="340" y="20" width="220" height="55" rx="10" fill="#eef4ff" stroke="#3b5bdb"/>
  <text x="450" y="53" text-anchor="middle" font-size="16" font-family="Arial">silver/orders</text>
  <line x1="450" y1="75" x2="180" y2="135" stroke="#333" stroke-width="2"/>
  <line x1="450" y1="75" x2="450" y2="135" stroke="#333" stroke-width="2"/>
  <line x1="450" y1="75" x2="720" y2="135" stroke="#333" stroke-width="2"/>
  <rect x="60" y="135" width="240" height="60" rx="10" fill="#fff4e6" stroke="#f08c00"/>
  <text x="180" y="170" text-anchor="middle" font-size="15" font-family="Arial">order_date=2026-08-10</text>
  <rect x="330" y="135" width="240" height="60" rx="10" fill="#fff4e6" stroke="#f08c00"/>
  <text x="450" y="170" text-anchor="middle" font-size="15" font-family="Arial">order_date=2026-08-11</text>
  <rect x="600" y="135" width="240" height="60" rx="10" fill="#ebfbee" stroke="#2b8a3e"/>
  <text x="720" y="170" text-anchor="middle" font-size="15" font-family="Arial">order_date=2026-08-12</text>
  <text x="720" y="240" text-anchor="middle" font-size="14" font-family="Arial">Query reads only this folder</text>
</svg>

## Simple definition

```text
Partitioning = storing files in folders based on useful column values.
```


# 10. Real-Life Partitioning Examples

| Scenario | Common Query | Good Partition Columns |
|---|---|---|
| E-commerce orders | Yesterday's revenue | order_date |
| Food delivery | Delhi orders for yesterday | order_date, city |
| Banking transactions | India transactions for a date | transaction_date, country |
| Clickstream events | Purchase events for a date | event_date, event_type |
| IoT sensor data | Region events for a date | sensor_date, region |
| Healthcare visits | Department visits by day | visit_date, department |

## Good partition columns

```text
date
month
country
region
city
event_type
department
source_system
```

## Bad partition columns

```text
order_id
transaction_id
customer_id
email
phone_number
timestamp
uuid
session_id
```

## Rule

```text
Partition by columns that appear often in filters and have a manageable number of values.
```


# 11. Write Partitioned Parquet Data

We will write Silver orders partitioned by `order_date`.

This is a common pattern for daily batch pipelines.


In [ ]:
partitioned_orders_path = silver_path / "orders_partitioned_by_date"

silver_orders.write.mode("overwrite").partitionBy("order_date").parquet(str(partitioned_orders_path))

for path in partitioned_orders_path.rglob("*"):
    if path.is_dir():
        print(path)


datalake_storage_delta_bridge/lake/silver/orders_partitioned_by_date/order_date=2026-08-12
datalake_storage_delta_bridge/lake/silver/orders_partitioned_by_date/order_date=2026-08-11
datalake_storage_delta_bridge/lake/silver/orders_partitioned_by_date/order_date=2026-08-10
datalake_storage_delta_bridge/lake/silver/orders_partitioned_by_date/order_date=2026-08-13


# 12. Partition Pruning

Partition pruning means Spark skips partition folders that are not needed.

If data is partitioned by `order_date`, this query can read only one date folder:

```python
orders.filter(col("order_date") == "2026-08-12")
```

This improves:

```text
query speed
data scan size
cloud cost
```


In [ ]:
partitioned_orders = spark.read.parquet(str(partitioned_orders_path))

one_day_orders = partitioned_orders.filter(col("order_date") == "2026-08-12")

one_day_orders.show()
one_day_orders.explain(True)


+--------+-----------+----------+------+---------+---------+-------+--------------------+----------+
|order_id|customer_id|product_id|amount|   status|     city|country|        processed_at|order_date|
+--------+-----------+----------+------+---------+---------+-------+--------------------+----------+
|    1007|       C002|      P005|  4500|completed|Bangalore|  India|2026-08-12 12:25:...|2026-08-12|
|    1008|       C003|      P003|   900|   failed|    Delhi|  India|2026-08-12 12:25:...|2026-08-12|
|    1009|       C006|      P006|  7500|completed|    Dubai|    UAE|2026-08-12 12:25:...|2026-08-12|
+--------+-----------+----------+------+---------+---------+-------+--------------------+----------+

== Parsed Logical Plan ==
'Filter '`=`('order_date, 2026-08-12)
+- Relation [order_id#576,customer_id#577,product_id#578,amount#579,status#580,city#581,country#582,processed_at#583,order_date#584] parquet

== Analyzed Logical Plan ==
order_id: int, customer_id: string, product_id: string, am

# 13. Multi-Column Partitioning

Sometimes one column is not enough.

Example:

```text
orders/
  order_date=2026-08-12/
    country=India/
    country=UAE/
```

This helps if queries often filter by both date and country.


In [ ]:
partitioned_by_date_country_path = silver_path / "orders_partitioned_by_date_country"

silver_orders.write.mode("overwrite").partitionBy("order_date", "country").parquet(
    str(partitioned_by_date_country_path)
)

for path in partitioned_by_date_country_path.rglob("*"):
    if path.is_dir():
        print(path)


datalake_storage_delta_bridge/lake/silver/orders_partitioned_by_date_country/order_date=2026-08-12
datalake_storage_delta_bridge/lake/silver/orders_partitioned_by_date_country/order_date=2026-08-11
datalake_storage_delta_bridge/lake/silver/orders_partitioned_by_date_country/order_date=2026-08-10
datalake_storage_delta_bridge/lake/silver/orders_partitioned_by_date_country/order_date=2026-08-13
datalake_storage_delta_bridge/lake/silver/orders_partitioned_by_date_country/order_date=2026-08-12/country=UAE
datalake_storage_delta_bridge/lake/silver/orders_partitioned_by_date_country/order_date=2026-08-12/country=India
datalake_storage_delta_bridge/lake/silver/orders_partitioned_by_date_country/order_date=2026-08-11/country=India
datalake_storage_delta_bridge/lake/silver/orders_partitioned_by_date_country/order_date=2026-08-10/country=India
datalake_storage_delta_bridge/lake/silver/orders_partitioned_by_date_country/order_date=2026-08-13/country=India
datalake_storage_delta_bridge/lake/silver

In [ ]:
orders_date_country = spark.read.parquet(str(partitioned_by_date_country_path))

india_orders_on_date = orders_date_country.filter(
    (col("order_date") == "2026-08-12") &
    (col("country") == "India")
)

india_orders_on_date.show()
india_orders_on_date.explain(True)


+--------+-----------+----------+------+---------+---------+--------------------+----------+-------+
|order_id|customer_id|product_id|amount|   status|     city|        processed_at|order_date|country|
+--------+-----------+----------+------+---------+---------+--------------------+----------+-------+
|    1007|       C002|      P005|  4500|completed|Bangalore|2026-08-12 12:25:...|2026-08-12|  India|
|    1008|       C003|      P003|   900|   failed|    Delhi|2026-08-12 12:25:...|2026-08-12|  India|
+--------+-----------+----------+------+---------+---------+--------------------+----------+-------+

== Parsed Logical Plan ==
'Filter 'and('`=`('order_date, 2026-08-12), '`=`('country, India))
+- Relation [order_id#693,customer_id#694,product_id#695,amount#696,status#697,city#698,processed_at#699,order_date#700,country#701] parquet

== Analyzed Logical Plan ==
order_id: int, customer_id: string, product_id: string, amount: int, status: string, city: string, processed_at: timestamp, order_

# 14. Bad Partitioning and Small Files Problem

Partitioning by a high-cardinality column creates too many folders.

Bad examples:

```text
order_id
customer_id
transaction_id
timestamp
uuid
```

If a table has 10 million orders and we partition by `order_id`, we may create millions of tiny folders.

This creates the small files problem.


In [ ]:
bad_partition_path = silver_path / "bad_partition_by_order_id"

silver_orders.write.mode("overwrite").partitionBy("order_id").parquet(str(bad_partition_path))

for path in bad_partition_path.rglob("*"):
    if path.is_dir():
        print(path)


datalake_storage_delta_bridge/lake/silver/bad_partition_by_order_id/order_id=1001
datalake_storage_delta_bridge/lake/silver/bad_partition_by_order_id/order_id=1004
datalake_storage_delta_bridge/lake/silver/bad_partition_by_order_id/order_id=1005
datalake_storage_delta_bridge/lake/silver/bad_partition_by_order_id/order_id=1009
datalake_storage_delta_bridge/lake/silver/bad_partition_by_order_id/order_id=1010
datalake_storage_delta_bridge/lake/silver/bad_partition_by_order_id/order_id=1008
datalake_storage_delta_bridge/lake/silver/bad_partition_by_order_id/order_id=1011
datalake_storage_delta_bridge/lake/silver/bad_partition_by_order_id/order_id=1007


In [ ]:
def count_data_files(path):
    path = Path(path)
    return len([
        p for p in path.rglob("*")
        if p.is_file() and not p.name.startswith("_") and not p.name.startswith(".")
    ])

print("Files in date partitioned table:", count_data_files(partitioned_orders_path))
print("Files in order_id partitioned table:", count_data_files(bad_partition_path))


Files in date partitioned table: 4
Files in order_id partitioned table: 8


# 15. Gold Layer Analytics

Gold tables answer business questions.

Examples:

```text
revenue by city
revenue by category
payment success report
customer summary
```


In [ ]:
orders_products = (
    silver_orders.alias("o")
    .join(silver_products.alias("p"), col("o.product_id") == col("p.product_id"), "left")
)

gold_revenue_by_city = (
    silver_orders
    .filter(col("status") == "completed")
    .groupBy("order_date", "city", "country")
    .agg(
        spark_sum("amount").alias("total_revenue"),
        count("*").alias("completed_order_count")
    )
    .orderBy("order_date", desc("total_revenue"))
)

gold_revenue_by_category = (
    orders_products
    .filter(col("status") == "completed")
    .groupBy("order_date", "category")
    .agg(
        spark_sum("amount").alias("total_revenue"),
        count("*").alias("completed_order_count")
    )
    .orderBy("order_date", desc("total_revenue"))
)

gold_revenue_by_city.show()
gold_revenue_by_category.show()


+----------+---------+---------+-------------+---------------------+
|order_date|     city|  country|total_revenue|completed_order_count|
+----------+---------+---------+-------------+---------------------+
|2026-08-10|    Delhi|    India|         2500|                    1|
|2026-08-11|     Pune|    India|         3200|                    1|
|2026-08-12|    Dubai|      UAE|         7500|                    1|
|2026-08-12|Bangalore|    India|         4500|                    1|
|2026-08-13|Singapore|Singapore|         6200|                    1|
|2026-08-13|    Delhi|    India|         2400|                    1|
+----------+---------+---------+-------------+---------------------+

+----------+-----------+-------------+---------------------+
|order_date|   category|total_revenue|completed_order_count|
+----------+-----------+-------------+---------------------+
|2026-08-10|electronics|         2500|                    1|
|2026-08-11|       NULL|         3200|                    1|
|202

In [ ]:
gold_revenue_by_city_path = gold_path / "revenue_by_city"
gold_revenue_by_category_path = gold_path / "revenue_by_category"

gold_revenue_by_city.write.mode("overwrite").partitionBy("order_date").parquet(str(gold_revenue_by_city_path))
gold_revenue_by_category.write.mode("overwrite").partitionBy("order_date").parquet(str(gold_revenue_by_category_path))

print("Gold tables written.")


Gold tables written.


# 16. Why Partitioning Alone is Not Enough

Partitioning improves read performance.

But partitioning does not solve reliability problems.

Plain Parquet Data Lake problems:

```text
pipeline fails halfway while writing
two jobs write to the same folder
schema changes unexpectedly
need to update or delete records
need to upsert new payment status
need to query yesterday's table version
need to rollback after a bad load
```

This leads to Delta Lake.

## Key bridge

```text
Partitioning makes a Data Lake faster.
Delta Lake makes a Data Lake reliable.
```


# 17. What is Delta Lake?

Delta Lake is a storage layer that adds reliability to Data Lake files.

Plain Data Lake:

```text
Parquet files + folders
```

Delta Lake:

```text
Parquet files + _delta_log transaction log
```

<svg width="900" height="260" xmlns="http://www.w3.org/2000/svg">
  <rect x="50" y="60" width="300" height="150" rx="12" fill="#eef4ff" stroke="#3b5bdb"/>
  <text x="200" y="95" text-anchor="middle" font-size="18" font-family="Arial">Plain Data Lake</text>
  <text x="200" y="130" text-anchor="middle" font-size="14" font-family="Arial">Parquet files</text>
  <text x="200" y="160" text-anchor="middle" font-size="14" font-family="Arial">Partition folders</text>
  <rect x="550" y="60" width="300" height="150" rx="12" fill="#ebfbee" stroke="#2b8a3e"/>
  <text x="700" y="95" text-anchor="middle" font-size="18" font-family="Arial">Delta Lake</text>
  <text x="700" y="130" text-anchor="middle" font-size="14" font-family="Arial">Parquet files</text>
  <text x="700" y="160" text-anchor="middle" font-size="14" font-family="Arial">_delta_log</text>
  <line x1="350" y1="135" x2="550" y2="135" stroke="#333" stroke-width="2" marker-end="url(#d)"/>
  <text x="450" y="115" text-anchor="middle" font-size="14" font-family="Arial">add transaction log</text>
  <defs><marker id="d" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#333"/></marker></defs>
</svg>

## Simple definition

```text
Delta Lake makes Parquet files behave like reliable database tables.
```


# 18. Real-Life Delta Lake Problems and Solutions

## Problem 1: Failed daily pipeline

A job writes only half the files and fails.

Plain Parquet may expose partial data.

Delta Lake uses atomic commits so readers see only valid table versions.

## Problem 2: Payment status updates

Payment status changes from pending to success.

Plain Parquet updates are difficult.

Delta Lake supports update and merge operations.

## Problem 3: Schema change

A source system adds `discount_amount`.

Plain files may become inconsistent.

Delta Lake can enforce schema and control schema evolution.

## Problem 4: Debugging bad data

Today's dashboard is wrong.

Delta Lake can query previous versions using time travel.

## Problem 5: Concurrent writes

Two pipelines write at the same time.

Delta Lake uses transaction log commits to maintain consistency.


# 19. Delta Lake Feature Summary

| Feature | Why it matters |
|---|---|
| ACID transactions | Prevent partial or corrupt writes |
| Transaction log | Tracks table versions |
| Schema enforcement | Protects table structure |
| Schema evolution | Allows controlled changes |
| Update/delete/merge | Supports changing business data |
| Time travel | Query previous versions |
| Table history | Debug pipeline changes |
| Works with Parquet | Uses efficient columnar files underneath |

## Main idea

```text
A Data Lake stores files.
Delta Lake turns those files into trustworthy tables.
```


# 20. Install Delta Lake for PySpark

The next section runs a local Delta Lake demo.

If package installation fails in your environment, run this section later in Databricks or another Spark environment with Delta support.


In [ ]:
!pip install delta-spark==3.2.0 -q


In [ ]:
from delta import configure_spark_with_delta_pip

spark.stop()

builder = (
    SparkSession.builder
    .appName("DeltaLakeBridgeDemo")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark


ImportError: cannot import name '_to_seq' from 'pyspark.sql.column' (/usr/local/lib/python3.12/dist-packages/pyspark/sql/column.py)

# 21. Write Silver Orders as a Delta Table

Writing Delta looks similar to writing Parquet, but Delta creates a transaction log.

Plain Parquet:

```python
df.write.parquet(path)
```

Delta:

```python
df.write.format("delta").save(path)
```


In [ ]:
delta_base_path = base_path / "delta_lake"
delta_silver_orders_path = delta_base_path / "silver" / "orders_delta"

silver_orders.write.format("delta").mode("overwrite").save(str(delta_silver_orders_path))

print("Delta table path:", delta_silver_orders_path)

for path in delta_silver_orders_path.rglob("*"):
    print(path)


In [ ]:
orders_delta = spark.read.format("delta").load(str(delta_silver_orders_path))
orders_delta.show()
orders_delta.printSchema()


# 22. Delta Transaction Log

The `_delta_log` folder records table changes.

It tracks:

```text
which files belong to the table
which files were added
which files were removed
table version history
metadata changes
```

This log is what gives Delta Lake table reliability.


In [ ]:
delta_log_path = delta_silver_orders_path / "_delta_log"

for path in delta_log_path.rglob("*"):
    if path.is_file():
        print(path.name)


# 23. Append New Day Orders

A common real-life pipeline receives new data every day.

We will append new orders for a new date.


In [ ]:
new_orders = spark.createDataFrame([
    (1012, "C001", "P002", "2026-08-14", 1800, "completed", "Delhi", "India"),
    (1013, "C003", "P004", "2026-08-14", 3200, "completed", "Mumbai", "India"),
    (1014, "C006", "P006", "2026-08-14", 7500, "pending", "Dubai", "UAE"),
], ["order_id", "customer_id", "product_id", "order_date", "amount", "status", "city", "country"])

new_orders_clean = (
    new_orders
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("status", lower(trim(col("status"))))
    .withColumn("processed_at", current_timestamp())
)

new_orders_clean.write.format("delta").mode("append").save(str(delta_silver_orders_path))

spark.read.format("delta").load(str(delta_silver_orders_path)).orderBy("order_id").show()


# 24. Delta Table History and Time Travel

Delta keeps table history.

This helps with debugging and auditing.

Time travel means reading an older version of the table.


In [ ]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, str(delta_silver_orders_path))
delta_table.history().show(truncate=False)


In [ ]:
orders_version_0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(str(delta_silver_orders_path))
)

print("Rows in version 0:", orders_version_0.count())
orders_version_0.orderBy("order_id").show()

current_orders_delta = spark.read.format("delta").load(str(delta_silver_orders_path))
print("Rows in current version:", current_orders_delta.count())
current_orders_delta.orderBy("order_id").show()


# 25. Update, Delete, and Merge

Delta supports changing data in table form.

Real-life examples:

```text
Order status changes from pending to completed
Invalid test order must be deleted
New source file contains both inserts and updates
```


In [ ]:
# Update order status

delta_table.update(
    condition="order_id = 1014",
    set={"status": "'completed'"}
)

spark.read.format("delta").load(str(delta_silver_orders_path)).filter(col("order_id") == 1014).show()


In [ ]:
# Delete an invalid or failed order from trusted Silver table

delta_table.delete("order_id = 1008")

spark.read.format("delta").load(str(delta_silver_orders_path)).orderBy("order_id").show()


In [ ]:
# Merge / upsert example

new_record = spark.createDataFrame([
    (1015, "C008", "P003", "2026-08-15", 900, "completed", "Delhi", "India")
], ["order_id", "customer_id", "product_id", "order_date", "amount", "status", "city", "country"]) .withColumn("order_date", to_date(col("order_date"))) .withColumn("processed_at", current_timestamp())

status_update = (
    spark.read.format("delta").load(str(delta_silver_orders_path))
    .filter(col("order_id") == 1005)
    .withColumn("status", lit("completed"))
)

merge_source = status_update.unionByName(new_record)

merge_source.show()

delta_table.alias("target").merge(
    merge_source.alias("source"),
    "target.order_id = source.order_id"
).whenMatchedUpdateAll()  .whenNotMatchedInsertAll()  .execute()

spark.read.format("delta").load(str(delta_silver_orders_path)).orderBy("order_id").show()


# 26. Schema Enforcement Example

Delta Lake protects table schema.

If a new DataFrame has a different schema, Delta can reject it unless schema evolution is explicitly enabled.

This prevents accidental schema corruption.


In [ ]:
bad_schema_orders = spark.createDataFrame([
    (2001, "C001", 9999, "completed", "unexpected_column_value")
], ["order_id", "customer_id", "amount", "status", "new_unexpected_column"])

try:
    bad_schema_orders.write.format("delta").mode("append").save(str(delta_silver_orders_path))
except Exception as error:
    print("Delta rejected the write because of schema mismatch.")
    print(type(error).__name__)


# 27. Delta Lake with Bronze, Silver, Gold

A practical architecture:

```text
Bronze:
Raw CSV/JSON files

Silver:
Cleaned Delta tables

Gold:
Business-ready Delta tables
```

<svg width="900" height="240" xmlns="http://www.w3.org/2000/svg">
  <rect x="60" y="80" width="190" height="80" rx="12" fill="#fff4e6" stroke="#f08c00"/>
  <text x="155" y="112" text-anchor="middle" font-size="17" font-family="Arial">Bronze</text>
  <text x="155" y="138" text-anchor="middle" font-size="13" font-family="Arial">Raw CSV / JSON</text>
  <rect x="350" y="80" width="190" height="80" rx="12" fill="#f1f3f5" stroke="#495057"/>
  <text x="445" y="112" text-anchor="middle" font-size="17" font-family="Arial">Silver</text>
  <text x="445" y="138" text-anchor="middle" font-size="13" font-family="Arial">Clean Delta Tables</text>
  <rect x="640" y="80" width="190" height="80" rx="12" fill="#ebfbee" stroke="#2b8a3e"/>
  <text x="735" y="112" text-anchor="middle" font-size="17" font-family="Arial">Gold</text>
  <text x="735" y="138" text-anchor="middle" font-size="13" font-family="Arial">Analytics Delta Tables</text>
  <line x1="250" y1="120" x2="350" y2="120" stroke="#333" stroke-width="2" marker-end="url(#e)"/>
  <line x1="540" y1="120" x2="640" y2="120" stroke="#333" stroke-width="2" marker-end="url(#e)"/>
  <defs><marker id="e" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#333"/></marker></defs>
</svg>


In [ ]:
orders_delta_current = spark.read.format("delta").load(str(delta_silver_orders_path))

gold_delta_revenue_by_city = (
    orders_delta_current
    .filter(col("status") == "completed")
    .groupBy("order_date", "city", "country")
    .agg(
        spark_sum("amount").alias("total_revenue"),
        count("*").alias("completed_order_count")
    )
    .orderBy("order_date", desc("total_revenue"))
)

gold_delta_revenue_by_city.show()

gold_delta_revenue_path = delta_base_path / "gold" / "revenue_by_city_delta"
gold_delta_revenue_by_city.write.format("delta").mode("overwrite").save(str(gold_delta_revenue_path))

spark.read.format("delta").load(str(gold_delta_revenue_path)).show()


# 28. Plain Parquet vs Delta Lake

| Capability | Plain Parquet | Delta Lake |
|---|---:|---:|
| Efficient columnar storage | Yes | Yes |
| Partition pruning | Yes | Yes |
| ACID transactions | No | Yes |
| Transaction log | No | Yes |
| Update/delete | Difficult | Supported |
| Merge/upsert | Difficult | Supported |
| Schema enforcement | Limited | Stronger |
| Time travel | No | Yes |
| Table history | No | Yes |

## Strong summary

```text
Parquet is a file format.
Delta Lake is a reliable table layer built on Parquet files.
```


# 29. Practice Exercises

## Exercise 1: Partitioning decision

Choose good partition columns for these datasets:

| Dataset | Possible Columns |
|---|---|
| orders | order_date, customer_id, order_id, city |
| clickstream | event_date, event_type, session_id, event_time |
| bank_transactions | transaction_date, transaction_id, country, card_number |
| app_logs | log_date, service_name, error_id, message |

Explain why each chosen column is useful.

## Exercise 2: PySpark partitioning

Create a partitioned Parquet table for clickstream events using:

```text
event_date
event_type
```

## Exercise 3: Delta Lake reasoning

Match each situation with a Delta Lake feature:

```text
Pipeline fails halfway
Need to update payment status
Need to query last week's table version
Source sends a new unexpected column
Need to insert new records and update existing records
```


In [ ]:
# Exercise starter: create partitioned clickstream data

from pyspark.sql.functions import to_date

clickstream_with_date = silver_clickstream.withColumn(
    "event_date",
    to_date(col("event_time"))
)

clickstream_partitioned_path = silver_path / "clickstream_partitioned"

# Write your code here:
# clickstream_with_date.write.mode("overwrite").partitionBy(...).parquet(...)

clickstream_with_date.show()


# 30. Interview Questions

## Q1. What is partitioning in a Data Lake?

Partitioning means storing files in folders based on column values such as date, country, city, or event type so query engines can skip unnecessary data.

## Q2. What is partition pruning?

Partition pruning is when Spark reads only the required partition folders and skips irrelevant folders.

## Q3. What makes a good partition column?

A good partition column is frequently used in filters, has a manageable number of unique values, and reduces the amount of data scanned.

## Q4. Why should we avoid partitioning by order_id or customer_id?

These columns usually have too many unique values, creating too many folders and small files.

## Q5. What is the small files problem?

The small files problem happens when a Data Lake has too many tiny files. Spark spends too much time listing files, managing metadata, and launching tasks.

## Q6. Why is Parquet preferred over CSV for analytics?

Parquet is columnar, compressed, and efficient for analytical queries because engines can read only required columns.

## Q7. Why do we need Delta Lake if we already have Parquet?

Parquet is a file format. Delta Lake adds a transaction log, ACID transactions, updates, deletes, merge, schema enforcement, and time travel.

## Q8. What is the `_delta_log` folder?

The `_delta_log` folder stores the transaction history of a Delta table. It tracks table versions and file changes.

## Q9. What is time travel in Delta Lake?

Time travel lets users query older versions of a Delta table.

## Q10. How does Delta Lake connect to Databricks?

Delta Lake provides reliable lakehouse tables. Databricks provides a managed platform to build, run, schedule, govern, and query those tables.


# 31. Final Summary

## File formats

```text
CSV and JSON are common in Bronze.
Parquet is common in Silver and Gold.
```

## Partitioning

```text
Partitioning helps Spark read less data.
Good partition columns are commonly used filters with manageable unique values.
```

## Small files problem

```text
Too many tiny files make Data Lake processing slow.
```

## Delta Lake

```text
Delta Lake makes Data Lake tables reliable using Parquet files plus a transaction log.
```

## Strong final takeaway

```text
Partitioning makes a Data Lake faster.
Delta Lake makes a Data Lake reliable.
Databricks helps teams build and manage lakehouse pipelines on a platform.
```
